# 4.2 针对硬件的代码生成

通过编译器将高层计算图转换为目标硬件的高效代码。不同硬件架构有不同的计算特性和内存层次，需要针对性的代码生成策略。

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import time

torch.manual_seed(42)
print(f"PyTorch version: {torch.__version__}")

### TVM风格算子搜索

TVM通过搜索最优算子实现（AutoTVM/Ansor）为目标硬件生成高效代码。核心思想：同一算子在不同硬件上的最优实现不同，需要搜索最优参数配置。

In [ ]:
class MatMulTuningSpace:
    """模拟TVM的矩阵乘法调优空间"""
    def __init__(self, M, N, K):
        self.M = M
        self.N = N
        self.K = K
        self.tile_sizes = [(16, 16), (32, 32), (64, 64), (128, 128)]
        self.unroll_factors = [1, 2, 4, 8]
        self.vector_lengths = [1, 4, 8]

    def enumerate_configs(self):
        configs = []
        for tile in self.tile_sizes:
            for unroll in self.unroll_factors:
                for vec in self.vector_lengths:
                    configs.append({
                        'tile_m': tile[0], 'tile_n': tile[1],
                        'unroll': unroll, 'vector_len': vec
                    })
        return configs

    def estimate_performance(self, config):
        """模拟性能评估（实际TVM会在硬件上实测）"""
        tile_m = config['tile_m']
        tile_n = config['tile_n']
        unroll = config['unroll']
        vec = config['vector_len']

        cache_efficiency = min(1.0, (tile_m * tile_n) / (64 * 64))
        parallelism = min(1.0, unroll / 4)
        vector_efficiency = min(1.0, vec / 8)
        register_pressure = min(1.0, 1.0 / (tile_m * tile_n / 1024))

        score = cache_efficiency * 0.4 + parallelism * 0.2 + vector_efficiency * 0.3 + register_pressure * 0.1
        return score

M, N, K = 512, 512, 512
tuner = MatMulTuningSpace(M, N, K)
configs = tuner.enumerate_configs()

print(f"=== TVM风格算子调优空间 ===")
print(f"矩阵乘法: {M}x{K} @ {K}x{N}")
print(f"候选配置数: {len(configs)}")

results = [(c, tuner.estimate_performance(c)) for c in configs]
results.sort(key=lambda x: x[1], reverse=True)

print(f"\nTop-5配置:")
for config, score in results[:5]:
    print(f"  tile=({config['tile_m']},{config['tile_n']}) unroll={config['unroll']} vec={config['vector_len']} score={score:.4f}")

print(f"\nTVM调优流程:")
print(f"1. 定义算子计算逻辑")
print(f"2. 枚举可能的调度配置(tile/unroll/vectorize)")
print(f"3. 在目标硬件上实测每个配置")
print(f"4. 选择最优配置生成代码")

### ONNX Runtime 量化推理

In [ ]:
class ONNXStyleQuantizedMatMul(nn.Module):
    """模拟ONNX Runtime INT8量化矩阵乘法"""
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(out_dim, in_dim) * 0.1)
        self.bias = nn.Parameter(torch.randn(out_dim) * 0.01)

    def forward_fp32(self, x):
        return F.linear(x, self.weight, self.bias)

    def forward_int8(self, x):
        """INT8量化前向传播
        注意：本实现为教学演示，量化后仍转为float计算以简化逻辑。
        实际INT8推理应使用硬件原生INT8算术（如NPU/TPU的INT8 Tensor Core），
        此处仅演示量化/反量化的数学流程，不产生实际INT8加速效果。"""
        w_scale = self.weight.abs().max() / 127.0
        w_int8 = torch.round(self.weight / w_scale.clamp(min=1e-8)).clamp(-128, 127).to(torch.int8)
        x_scale = x.abs().max() / 127.0
        x_int8 = torch.round(x / x_scale.clamp(min=1e-8)).clamp(-128, 127).to(torch.int8)
        result_int32 = x_int8.float() @ w_int8.float().T
        result_fp32 = result_int32 * (x_scale.unsqueeze(-1) * w_scale)
        return result_fp32 + self.bias

in_dim, out_dim = 256, 128
q_matmul = ONNXStyleQuantizedMatMul(in_dim, out_dim)
x = torch.randn(64, in_dim)

with torch.no_grad():
    out_fp32 = q_matmul.forward_fp32(x)
    out_int8 = q_matmul.forward_int8(x)
    diff = (out_fp32 - out_int8).norm() / out_fp32.norm() * 100

n_iters = 500
with torch.no_grad():
    start = time.time()
    for _ in range(n_iters):
        _ = q_matmul.forward_fp32(x)
    fp32_time = time.time() - start

    start = time.time()
    for _ in range(n_iters):
        _ = q_matmul.forward_int8(x)
    int8_time = time.time() - start

print(f"=== ONNX Runtime风格INT8推理 ===")
print(f"FP32输出范数: {out_fp32.norm():.4f}")
print(f"INT8输出范数: {out_int8.norm():.4f}")
print(f"相对误差: {diff:.4f}%")
print(f"FP32时间: {fp32_time:.4f}s")
print(f"INT8时间: {int8_time:.4f}s")
print(f"\nONNX Runtime优化:")
print(f"1. INT8算子: 利用VNNI/DP4A等硬件指令")
print(f"2. 图级优化: 常量折叠/算子融合/死代码消除")
print(f"3. 执行提供器: CPU/CUDA/TensorRT/OpenVINO")

### 编译框架对比

In [ ]:
print(f"{'框架':<20} {'核心方法':<25} {'支持硬件':<25} {'特点':<20}")
print("-" * 90)
frameworks = [
    ("TVM/Apache TVM", "AutoTVM/Ansor搜索", "CPU/GPU/NPU", "跨平台，搜索最优kernel"),
    ("MLIR/XLA", "多级IR逐级lowering", "CPU/GPU/TPU", "编译器基础设施"),
    ("TensorRT", "层融合+精度校准", "NVIDIA GPU", "GPU推理极致优化"),
    ("Core ML Tools", "图优化+ANE编译", "Apple Silicon", "Apple生态最优"),
    ("ONNX Runtime", "图优化+量化+EP", "CPU/GPU/NPU", "通用推理引擎"),
    ("torch.compile", "TorchDynamo+Inductor", "CPU/GPU", "PyTorch原生编译"),
]
for name, method, hw, feature in frameworks:
    print(f"{name:<20} {method:<25} {hw:<25} {feature:<20}")

print(f"\n=== 产业级选择建议 ===")
print(f"1. NVIDIA GPU: TensorRT (最成熟，性能最优)")
print(f"2. Apple端侧: Core ML (ANE加速，生态集成)")
print(f"3. 通用CPU/NPU: ONNX Runtime 或 TVM")
print(f"4. PyTorch快速验证: torch.compile (开发效率高)")

### 产业级实战：真实 ONNX 导出 + ONNX Runtime 推理

ONNX 是产业界最通用的模型交换格式。以下展示完整的端到端流程：
PyTorch模型 → ONNX导出 → ONNX Runtime推理 → 性能对比

In [ ]:
import io
import time
import onnx
import onnxruntime as ort
from transformers import GPT2LMHeadModel, GPT2Config

config = GPT2Config(vocab_size=1000, n_positions=64, n_embd=128, n_layer=4, n_head=4)
gpt2_small = GPT2LMHeadModel(config)
gpt2_small.eval()

class GPT2OnnxWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, input_ids):
        return self.model(input_ids).logits

wrapper = GPT2OnnxWrapper(gpt2_small)
dummy_ids = torch.randint(0, 1000, (1, 32))

onnx_buffer = io.BytesIO()
torch.onnx.export(
    wrapper,
    dummy_ids,
    onnx_buffer,
    input_names=['input_ids'],
    output_names=['logits'],
    dynamic_axes={'input_ids': {0: 'batch', 1: 'seq_len'}, 'logits': {0: 'batch', 1: 'seq_len'}},
    opset_version=14,
    do_constant_folding=True,
)
onnx_buffer.seek(0)
onnx_model = onnx.load(onnx_buffer)
onnx.checker.check_model(onnx_model)
onnx_size = len(onnx_buffer.getvalue())

with torch.no_grad():
    pt_output = wrapper(dummy_ids).numpy()

sess = ort.InferenceSession(onnx_buffer.getvalue())
ort_output = sess.run(None, {'input_ids': dummy_ids.numpy()})[0]

max_diff = np.abs(pt_output - ort_output).max()

print(f"=== ONNX导出 + ONNX Runtime推理 ===")
print(f"模型: GPT-2 (4层, 128维, 1000词表)")
print(f"参数量: {sum(p.numel() for p in gpt2_small.parameters()):,}")
print(f"ONNX文件大小: {onnx_size/1024:.1f} KB")
print(f"PyTorch输出shape: {pt_output.shape}")
print(f"ONNX Runtime输出shape: {ort_output.shape}")
print(f"最大数值差异: {max_diff:.8f}")
print(f"数值一致性: {'PASS' if max_diff < 1e-5 else 'FAIL'}")

In [ ]:
print(f"=== 性能对比: PyTorch vs ONNX Runtime ===")

def benchmark_torch(model, input_ids, warmup=10, runs=50):
    with torch.no_grad():
        for _ in range(warmup):
            model(input_ids)
        start = time.perf_counter()
        for _ in range(runs):
            model(input_ids)
        return (time.perf_counter() - start) / runs * 1000

def benchmark_ort(session, input_ids_np, warmup=10, runs=50):
    for _ in range(warmup):
        session.run(None, {'input_ids': input_ids_np})
    start = time.perf_counter()
    for _ in range(runs):
        session.run(None, {'input_ids': input_ids_np})
    return (time.perf_counter() - start) / runs * 1000

batch_sizes = [1, 4, 8]
seq_len = 32

print(f"{'Batch':<8} {'PyTorch(ms)':<14} {'ORT(ms)':<14} {'加速比':<10}")
print("-" * 46)
for bs in batch_sizes:
    ids = torch.randint(0, 1000, (bs, seq_len))
    pt_time = benchmark_torch(wrapper, ids)
    ort_time = benchmark_ort(sess, ids.numpy())
    speedup = pt_time / ort_time
    print(f"{bs:<8} {pt_time:<14.2f} {ort_time:<14.2f} {speedup:<10.2f}x")

print(f"\nONNX Runtime优化选项:")
sess_opts = ort.SessionOptions()
sess_opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
sess_opts.intra_op_num_threads = 4
sess_optimized = ort.InferenceSession(onnx_buffer.getvalue(), sess_opts)

ids_b1 = torch.randint(0, 1000, (1, seq_len))
ort_opt_time = benchmark_ort(sess_optimized, ids_b1.numpy())
print(f"  默认ORT: {benchmark_ort(sess, ids_b1.numpy()):.2f} ms")
print(f"  优化ORT: {ort_opt_time:.2f} ms")

print(f"\n产业界ONNX部署最佳实践:")
print(f"1. 导出: torch.onnx.export + do_constant_folding=True + opset_version=14+")
print(f"2. 验证: onnx.checker.check_model + 数值一致性对比")
print(f"3. 优化: ORT_ENABLE_ALL + 线程数调优 + IO Binding")
print(f"4. 量化: onnxruntime.quantization.quantize_static (INT8)")
print(f"5. 常见问题: 动态shape → dynamic_axes; 自定义算子 → onnx-script注册")